# Batch Processing & Throughput with DistLLM

This notebook sends **many prompts** through the DistLLM chat completions API in a batch and measures end-to-end **throughput in tokens/second**.

You can run the batch sequentially (simplest) or concurrently with a thread pool to saturate the server. Throughput is computed from `response.usage.completion_tokens` divided by wall-clock time.

## Requirements

```bash
pip install openai distllm_sdk
```

Start the DistLLM API server before running the code cells (see the README).

In [ ]:
# ---------------------------------------------------------------------------
# Connection configuration
# ---------------------------------------------------------------------------
# Point the OpenAI-compatible client at a running DistLLM API.
# Start the server first, e.g.:
#     distllm-api --model distributed-llm --host 0.0.0.0 --port 8000
#
# The API key is not required for a local deployment; set DISTLLM_API_KEY if
# your deployment enforces authentication.
import os

BASE_URL = os.environ.get("DISTLLM_BASE_URL", "http://localhost:8000/v1")
API_KEY = os.environ.get("DISTLLM_API_KEY", "not-needed")
MODEL = os.environ.get("DISTLLM_MODEL", "distributed-llm")

print(f"Using endpoint: {BASE_URL}")
print(f"Model: {MODEL}")


In [ ]:
# ---------------------------------------------------------------------------
# Prepare a batch of prompts
# ---------------------------------------------------------------------------
prompts = [
    'Explain what a tensor is in one sentence.',
    'Name three popular open-weight language models.',
    'What is the difference between fine-tuning and RAG?',
    'Give a one-line definition of speculative decoding.',
    'What does a coordinator node do in DistLLM?',
    'Summarize the benefits of model sharding in five words.',
    'What is tokens-per-second a measure of?',
    'Write a tagline for a distributed inference engine.',
]
print(f'Prepared a batch of {len(prompts)} prompts.')


In [ ]:
# ---------------------------------------------------------------------------
# Sequential batch + throughput measurement
# ---------------------------------------------------------------------------
try:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
except ImportError:
    from distllm_sdk.compat import openai_compat as oc
    client = oc.OpenAI(base_url=BASE_URL, api_key=API_KEY)

import time

total_completion_tokens = 0
t0 = time.perf_counter()
for i, p in enumerate(prompts, 1):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': p}],
        max_tokens=64, temperature=0.3)
    text = resp.choices[0].message.content
    usage = getattr(resp, 'usage', None)
    ct = usage.completion_tokens if usage else 0
    total_completion_tokens += ct
    print(f'[{i}/{len(prompts)}] +{ct} tok | {text[:60].strip()!r}')
elapsed = time.perf_counter() - t0

tps = total_completion_tokens / elapsed if elapsed > 0 else 0.0
print('\n--- Sequential batch summary ---')
print(f'Prompts           : {len(prompts)}')
print(f'Completion tokens : {total_completion_tokens}')
print(f'Wall-clock time   : {elapsed:.2f}s')
print(f'Throughput        : {tps:.1f} tokens/sec')


In [ ]:
# ---------------------------------------------------------------------------
# Concurrent batch (thread pool) to increase throughput
# ---------------------------------------------------------------------------
# Guard the concurrency so the demo still works if futures are unavailable.
try:
    from concurrent.futures import ThreadPoolExecutor
    MAX_WORKERS = 8

    def _one(p):
        r = client.chat.completions.create(
            model=MODEL, messages=[{'role': 'user', 'content': p}],
            max_tokens=64, temperature=0.3)
        u = getattr(r, 'usage', None)
        return (u.completion_tokens if u else 0, r.choices[0].message.content)

    t0 = time.perf_counter()
    total = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        for res in ex.map(_one, prompts):
            total += res[0]
    elapsed = time.perf_counter() - t0
    tps = total / elapsed if elapsed > 0 else 0.0
    print('--- Concurrent batch summary ---')
    print(f'Workers           : {MAX_WORKERS}')
    print(f'Completion tokens : {total}')
    print(f'Wall-clock time   : {elapsed:.2f}s')
    print(f'Throughput        : {tps:.1f} tokens/sec')
except Exception as exc:  # pragma: no cover
    print(f'[!] Concurrent batch skipped: {exc}')
